# LeetCode 1725 — Number Of Rectangles That Can Form The Largest Square bằng Python

**Link đề:** https://leetcode.com/problems/number-of-rectangles-that-can-form-the-largest-square/
**Độ khó:** Easy · **Chủ đề:** Array

```python
class Solution:
    def countGoodRectangles(self, rectangles: List[List[int]]) -> int:
        ...
```

## Mục tiêu học
1. Biến bài hình học thành bài mảng: hình vuông lớn nhất cắt từ `[l, w]` có cạnh **`min(l, w)`**.
2. Nắm công cụ Python: unpacking `l, w`, `min`/`max`, list comprehension, `list.count`, `collections.Counter`.
3. Lời giải **hai lượt** (tìm max rồi đếm) → **một lượt** (cập nhật max và bộ đếm cùng lúc, reset khi có max mới).
4. Trace trạng thái `(max_len, count)` qua từng hình; chứng minh invariant.
5. Nhận diện pattern "đếm số phần tử bằng giá trị lớn nhất" — dùng lại ở rất nhiều bài.

## 1. Đề bài và ví dụ

Cho mảng `rectangles` với `rectangles[i] = [l_i, w_i]` là chiều dài và chiều rộng của hình chữ nhật thứ `i`. Có thể cắt từ hình `i` một **hình vuông cạnh `k`** nếu `k <= l_i` và `k <= w_i`. Ví dụ `[4, 6]` cắt được hình vuông cạnh tối đa `4`.

Gọi `maxLen` là cạnh của hình vuông **lớn nhất** cắt được từ **bất kỳ** hình nào. Trả về **số hình chữ nhật** cắt được hình vuông cạnh `maxLen`.

| `rectangles` | cạnh vuông lớn nhất mỗi hình | maxLen | Kết quả |
|---|---|:---:|:---:|
| `[[5,8],[3,9],[5,12],[16,5]]` | `[5, 3, 5, 5]` | 5 | `3` |
| `[[2,3],[3,7],[4,3],[3,7]]` | `[2, 3, 3, 3]` | 3 | `3` |

**Constraints:** `1 <= len(rectangles) <= 1000`; `rectangles[i].length == 2`; `1 <= l_i, w_i <= 10^9`; `l_i != w_i`.

### 1.1. Vì sao cạnh vuông lớn nhất là `min(l, w)`?

Cạnh `k` phải **không vượt** cả hai chiều ⇒ `k <= min(l, w)`. Và `k = min(l, w)` luôn cắt được (đặt hình vuông sát một góc). Vậy mỗi hình chữ nhật "rút gọn" thành **một con số** `min(l, w)`.

In [1]:
from typing import List

for l, w in [[4, 6], [5, 8], [16, 5], [3, 9]]:
    print(f"[{l}, {w}] -> cạnh vuông lớn nhất = min({l}, {w}) = {min(l, w)}")

[4, 6] -> cạnh vuông lớn nhất = min(4, 6) = 4
[5, 8] -> cạnh vuông lớn nhất = min(5, 8) = 5
[16, 5] -> cạnh vuông lớn nhất = min(16, 5) = 5
[3, 9] -> cạnh vuông lớn nhất = min(3, 9) = 3


## 2. Nền tảng Python cần biết

### 2.1. Unpacking cặp và `min`, `max`

`for l, w in rectangles:` tách mỗi `[l, w]` thành hai biến. `min(a, b)` / `max(a, b)` lấy nhỏ/lớn hơn; `max(list)` lấy lớn nhất của cả list.

In [2]:
rects = [[5, 8], [3, 9], [5, 12], [16, 5]]
for l, w in rects:
    print("l =", l, "w =", w, "-> min =", min(l, w))
print("max([5, 3, 5, 5]) =", max([5, 3, 5, 5]))

l = 5 w = 8 -> min = 5
l = 3 w = 9 -> min = 3
l = 5 w = 12 -> min = 5
l = 16 w = 5 -> min = 5
max([5, 3, 5, 5]) = 5


### 2.2. List comprehension — biến đổi cả mảng trong một dòng

`[min(l, w) for l, w in rects]` tạo list mới gồm cạnh vuông của từng hình. Đọc: "với mỗi `l, w` trong `rects`, lấy `min(l, w)`".

In [3]:
sides = [min(l, w) for l, w in rects]
print("sides =", sides)

sides = [5, 3, 5, 5]


### 2.3. `list.count(x)` — đếm số lần xuất hiện

`sides.count(5)` đếm bao nhiêu phần tử bằng `5`. Tốn `O(n)` vì phải duyệt cả list.

In [4]:
print("sides.count(5) =", sides.count(5))
print("sides.count(3) =", sides.count(3))
print("sides.count(99) =", sides.count(99))

sides.count(5) = 3
sides.count(3) = 1
sides.count(99) = 0


### 2.4. `collections.Counter` — đếm tất cả cùng lúc

`Counter(sides)` là dict `giá trị -> số lần`. `cnt[max(cnt)]` lấy số lần của **key lớn nhất** (`max` trên Counter duyệt các key).

In [5]:
from collections import Counter

cnt = Counter(sides)
print(cnt)
print("key lớn nhất:", max(cnt), "| số lần:", cnt[max(cnt)])

Counter({5: 3, 3: 1})
key lớn nhất: 5 | số lần: 3


## 3. Cách trực giác — hai lượt: tìm max rồi đếm

1. Tính `sides = [min(l, w) for l, w in rectangles]`.
2. `max_len = max(sides)`.
3. Đếm `sides.count(max_len)`.

In [6]:
def count_good_two_pass(rectangles: List[List[int]]) -> int:
    sides = [min(l, w) for l, w in rectangles]
    max_len = max(sides)
    return sides.count(max_len)

print(count_good_two_pass([[5, 8], [3, 9], [5, 12], [16, 5]]))   # 3
print(count_good_two_pass([[2, 3], [3, 7], [4, 3], [3, 7]]))     # 3

3
3


### Chi phí và hướng gọn hơn

Ba lượt `O(n)` (tạo `sides`, `max`, `count`) ⇒ tổng `O(n)`, bộ nhớ `O(n)` cho `sides`. Đã đủ tốt với `n ≤ 1000`. Nhưng có thể làm **một lượt, bộ nhớ `O(1)`**: vừa đi vừa giữ "max hiện tại" và "số hình đạt max hiện tại".

Đây không phải nút thắt về tốc độ mà là bài học về **duy trì trạng thái** — rất hữu ích khi dữ liệu đến dạng **luồng (stream)** và không lưu lại được.

## 4. Insight — cập nhật `(max_len, count)` trong một lượt

Với mỗi cạnh mới `s`:
- `s > max_len` → tìm được max **mới**: `max_len = s`, **`count = 1`** (reset — các hình cũ không còn đạt max).
- `s == max_len` → thêm một hình đạt max: `count += 1`.
- `s < max_len` → bỏ qua.

In [7]:
max_len, count = 0, 0
for s in [5, 3, 5, 5]:
    if s > max_len:
        max_len, count = s, 1
    elif s == max_len:
        count += 1
print("max_len =", max_len, "| count =", count)

max_len = 5 | count = 3


Cú pháp `max_len, count = s, 1` là **gán đồng thời** nhiều biến (tuple unpacking).

## 5. Lời giải chính — một lượt

**Đoạn 1:** `max_len = 0`, `count = 0` (mọi cạnh ≥ 1 nên 0 là giá trị khởi tạo an toàn).
**Đoạn 2:** duyệt `l, w`, tính `s = min(l, w)`, áp dụng 3 nhánh ở trên.
**Đoạn 3:** trả `count`.

In [8]:
def count_good_one_pass(rectangles: List[List[int]]) -> int:
    max_len = 0
    count = 0
    for l, w in rectangles:
        s = min(l, w)
        if s > max_len:          # max mới -> reset bộ đếm
            max_len = s
            count = 1
        elif s == max_len:       # thêm một hình đạt max
            count += 1
    return count

print(count_good_one_pass([[5, 8], [3, 9], [5, 12], [16, 5]]))   # 3
print(count_good_one_pass([[2, 3], [3, 7], [4, 3], [3, 7]]))     # 3

3
3


## 6. TRACE từng bước

Ví dụ 2: `[[2,3],[3,7],[4,3],[3,7]]`.

| i | [l, w] | s = min | so với max_len | max_len sau | count sau |
|---:|---|---:|---|---:|---:|
| 0 | [2, 3] | 2 | 2 > 0 → max mới | 2 | 1 |
| 1 | [3, 7] | 3 | 3 > 2 → max mới (reset) | 3 | 1 |
| 2 | [4, 3] | 3 | 3 == 3 → +1 | 3 | 2 |
| 3 | [3, 7] | 3 | 3 == 3 → +1 | 3 | 3 |

Chú ý bước `i = 1`: `count` bị **reset về 1** — hình `[2,3]` không còn tính nữa.

In [9]:
def trace_good(rectangles: List[List[int]]) -> int:
    max_len, count = 0, 0
    for i, (l, w) in enumerate(rectangles):          # enumerate + unpack lồng
        s = min(l, w)
        if s > max_len:
            note = f"{s} > {max_len} -> max mới, reset count"
            max_len, count = s, 1
        elif s == max_len:
            note = f"{s} == {max_len} -> count + 1"
            count += 1
        else:
            note = f"{s} < {max_len} -> bỏ qua"
        print(f"i={i}: [{l}, {w}] s={s:<3} | {note:<32} | max_len={max_len}, count={count}")
    return count

trace_good([[2, 3], [3, 7], [4, 3], [3, 7]])
print()
trace_good([[5, 8], [3, 9], [5, 12], [16, 5]])

i=0: [2, 3] s=2   | 2 > 0 -> max mới, reset count    | max_len=2, count=1
i=1: [3, 7] s=3   | 3 > 2 -> max mới, reset count    | max_len=3, count=1
i=2: [4, 3] s=3   | 3 == 3 -> count + 1              | max_len=3, count=2
i=3: [3, 7] s=3   | 3 == 3 -> count + 1              | max_len=3, count=3

i=0: [5, 8] s=5   | 5 > 0 -> max mới, reset count    | max_len=5, count=1
i=1: [3, 9] s=3   | 3 < 5 -> bỏ qua                  | max_len=5, count=1
i=2: [5, 12] s=5   | 5 == 5 -> count + 1              | max_len=5, count=2
i=3: [16, 5] s=5   | 5 == 5 -> count + 1              | max_len=5, count=3


3

`for i, (l, w) in enumerate(rectangles)`: `enumerate` cho cặp `(i, [l, w])`, và `(l, w)` tách tiếp phần tử thứ hai.

## 7. Các lời giải khác

### 7.1. `Counter` trên các cạnh

In [10]:
def count_good_counter(rectangles: List[List[int]]) -> int:
    cnt = Counter(min(l, w) for l, w in rectangles)
    return cnt[max(cnt)]

print(count_good_counter([[5, 8], [3, 9], [5, 12], [16, 5]]), count_good_counter([[2, 3], [3, 7], [4, 3], [3, 7]]))

3 3


### 7.2. Sắp xếp giảm dần rồi đếm phần tử đầu

Sau khi sắp giảm, max nằm ở đầu; đếm các phần tử liên tiếp bằng nó rồi dừng.

In [11]:
def count_good_sort(rectangles: List[List[int]]) -> int:
    sides = sorted((min(l, w) for l, w in rectangles), reverse=True)
    count = 0
    for s in sides:
        if s != sides[0]:
            break
        count += 1
    return count

print(count_good_sort([[5, 8], [3, 9], [5, 12], [16, 5]]), count_good_sort([[2, 3], [3, 7], [4, 3], [3, 7]]))

3 3


### 7.3. Hai lượt không tạo list phụ (generator)

Dùng generator expression hai lần — không lưu `sides`, bộ nhớ `O(1)`, đổi lại tính `min` hai lần mỗi hình.

In [12]:
def count_good_generator(rectangles: List[List[int]]) -> int:
    max_len = max(min(r) for r in rectangles)              # min(r) với r = [l, w]
    return sum(1 for r in rectangles if min(r) == max_len)

print("min([16, 5]) =", min([16, 5]))
print(count_good_generator([[5, 8], [3, 9], [5, 12], [16, 5]]))

min([16, 5]) = 5
3


### 7.4. Bảng so sánh

| Cách | Ý tưởng | Time | Space phụ | Ghi chú |
|---|---|---|---|---|
| Hai lượt + `count` | `sides`, `max`, `count` | `O(n)` | `O(n)` | trực giác |
| **Một lượt** | giữ `(max_len, count)` | `O(n)` | `O(1)` | khuyên dùng, hợp stream |
| `Counter` | đếm mọi cạnh | `O(n)` | `O(n)` | tổng quát, cho thêm thông tin |
| Sắp xếp | max ở đầu | `O(n log n)` | `O(n)` | chậm hơn, không cần thiết |
| Generator 2 lượt | không lưu list | `O(n)` | `O(1)` | gọn |

## 8. Code nộp LeetCode

In [13]:
class Solution:
    def countGoodRectangles(self, rectangles: List[List[int]]) -> int:
        max_len, count = 0, 0
        for l, w in rectangles:
            s = min(l, w)
            if s > max_len:
                max_len, count = s, 1
            elif s == max_len:
                count += 1
        return count

sol = Solution()
print(sol.countGoodRectangles([[5, 8], [3, 9], [5, 12], [16, 5]]), sol.countGoodRectangles([[2, 3], [3, 7], [4, 3], [3, 7]]))

3 3


Bản ngắn gọn:

In [14]:
class SolutionShort:
    def countGoodRectangles(self, rectangles: List[List[int]]) -> int:
        sides = [min(r) for r in rectangles]
        return sides.count(max(sides))

print(SolutionShort().countGoodRectangles([[5, 8], [3, 9], [5, 12], [16, 5]]))

3


## 9. Vì sao đúng?

**Bước rút gọn:** hình `i` cắt được hình vuông cạnh `maxLen` ⇔ `maxLen <= min(l_i, w_i)` ⇔ `s_i >= maxLen`. Mà `maxLen = max(s_j)` nên `s_i <= maxLen`. Vậy điều kiện ⇔ **`s_i == maxLen`**. Bài toán đúng bằng "đếm số phần tử bằng max của dãy `s`".

**Invariant của vòng một lượt:** sau khi xử lý `t` hình đầu, `max_len = max(s_0..s_{t-1})` và `count` = số chỉ số `j < t` với `s_j == max_len`.
- Ban đầu `t = 0`: quy ước `max_len = 0`, `count = 0` (dãy rỗng; mọi `s ≥ 1` nên hình đầu chắc chắn kích hoạt nhánh "max mới").
- Nhánh `s > max_len`: max mới là `s`; chỉ hình hiện tại đạt nó ⇒ `count = 1` ✓.
- Nhánh `s == max_len`: max không đổi, thêm một hình đạt ⇒ `count + 1` ✓.
- Nhánh `s < max_len`: không đổi gì ✓.

Khi `t = n` ta có đáp án.

## 10. Độ phức tạp

`n = len(rectangles)`.
- **Một lượt:** **Time `O(n)`**, **Space `O(1)`**.
- **Hai lượt / Counter:** Time `O(n)`, Space `O(n)`.
- **Sắp xếp:** Time `O(n log n)`.
- Giá trị tới `10^9` không ảnh hưởng (chỉ so sánh, Python không tràn số).

In [15]:
import random, time

random.seed(1725)
big = [[random.randint(1, 10**9), random.randint(1, 10**9)] for _ in range(10**5)]
for name, f in [("hai lượt", count_good_two_pass), ("một lượt", count_good_one_pass),
                ("Counter", count_good_counter), ("sort", count_good_sort)]:
    t = time.perf_counter()
    r = f(big)
    print(f"{name:>9}: {r}, {time.perf_counter() - t:.4f}s")

 hai lượt: 1, 0.0071s
 một lượt: 1, 0.0072s
  Counter: 1, 0.0182s


     sort: 1, 0.0253s


## 11. Bộ test

1. Hai ví dụ đề.
2. Biên: 1 hình; mọi hình cùng cạnh vuông.
3. Dễ sai: max mới xuất hiện **sau** nhiều hình (kiểm tra reset); max ở hình đầu tiên; giá trị lớn `10^9`; `l > w` và `l < w` lẫn lộn.
4. Đối chiếu 7 cài đặt trên dữ liệu ngẫu nhiên.

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

S = Solution()
check("đề 1", S.countGoodRectangles([[5, 8], [3, 9], [5, 12], [16, 5]]), 3)
check("đề 2", S.countGoodRectangles([[2, 3], [3, 7], [4, 3], [3, 7]]), 3)
check("1 hình", S.countGoodRectangles([[1, 2]]), 1)
check("cùng cạnh", S.countGoodRectangles([[4, 7], [9, 4], [4, 5]]), 3)
check("reset muộn", S.countGoodRectangles([[2, 3], [2, 5], [2, 9], [6, 7]]), 1)
check("max ở đầu", S.countGoodRectangles([[8, 9], [2, 3], [9, 8]]), 2)
check("giá trị lớn", S.countGoodRectangles([[10**9, 10**9 - 1], [10**9 - 1, 10**9]]), 2)

PASS đề 1: 3
PASS đề 2: 3
PASS 1 hình: 1
PASS cùng cạnh: 3
PASS reset muộn: 1
PASS max ở đầu: 2
PASS giá trị lớn: 2


In [17]:
impls = [count_good_two_pass, count_good_one_pass, count_good_counter, count_good_sort,
         count_good_generator, Solution().countGoodRectangles, SolutionShort().countGoodRectangles]
random.seed(42)
for _ in range(3000):
    n = random.randint(1, 15)
    rects = []
    for _ in range(n):
        l = random.randint(1, 8)
        w = random.choice([x for x in range(1, 9) if x != l])     # l != w theo đề
        rects.append([l, w])
    res = {f(rects) for f in impls}
    assert len(res) == 1, (rects, res)
print("Đối chiếu 7 cài đặt trên 3000 bộ ngẫu nhiên: PASS")

Đối chiếu 7 cài đặt trên 3000 bộ ngẫu nhiên: PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1 — Dùng `max(l, w)` thay vì `min(l, w)`

In [18]:
def wrong_max_side(rects):
    sides = [max(l, w) for l, w in rects]          # SAI: hình vuông phải vừa cả hai chiều
    return sides.count(max(sides))

r = [[5, 8], [3, 9], [5, 12], [16, 5]]
print("sai:", wrong_max_side(r), "| đúng:", count_good_one_pass(r))

sai: 1 | đúng: 3


**Sửa:** cạnh vuông bị giới hạn bởi chiều **nhỏ hơn** ⇒ `min`.

### Lỗi 2 — Quên reset `count` khi có max mới

In [19]:
def wrong_no_reset(rects):
    max_len, count = 0, 0
    for l, w in rects:
        s = min(l, w)
        if s > max_len:
            max_len = s
            count += 1          # SAI: phải là count = 1
        elif s == max_len:
            count += 1
    return count

r = [[2, 3], [3, 7], [4, 3], [3, 7]]
print("sai:", wrong_no_reset(r), "| đúng:", count_good_one_pass(r))

sai: 4 | đúng: 3


**Sửa:** `count = 1` khi tìm được max mới.

### Lỗi 3 — Dùng `if` thay vì `elif` → đếm trùng

In [20]:
def wrong_if_if(rects):
    max_len, count = 0, 0
    for l, w in rects:
        s = min(l, w)
        if s > max_len:
            max_len, count = s, 1
        if s == max_len:        # SAI: vừa gán max_len = s ở trên nên điều kiện này luôn đúng
            count += 1
    return count

print("sai:", wrong_if_if(r), "| đúng:", count_good_one_pass(r))

sai: 4 | đúng: 3


Sau nhánh đầu, `max_len == s` nên nhánh thứ hai **cũng chạy** → mỗi max mới bị tính 2. **Sửa:** dùng `elif`.

### Lỗi 4 — Đếm hình có **diện tích** lớn nhất

Đọc lướt "largest square" rồi so `l * w`. Đề hỏi **cạnh hình vuông cắt được**, không phải diện tích hình chữ nhật. `[16, 5]` có diện tích 80 nhưng chỉ cắt được vuông cạnh 5.

## 13. Mẹo nhận dạng pattern

- "Đếm số phần tử bằng giá trị lớn nhất / nhỏ nhất" → **một lượt** với cặp `(best, count)` và reset khi có `best` mới.
- Bài hình học đơn giản → tìm cách **rút mỗi đối tượng về một con số** (`min(l, w)`), rồi giải bài mảng.
- Bài có dữ liệu dạng luồng / không lưu được → duy trì trạng thái nhỏ thay vì lưu cả mảng.
- Bài liên quan: 1732 Find the Highest Altitude, 1672 Richest Customer Wealth (max của tổng), 414 Third Maximum Number (giữ nhiều trạng thái), 169 Majority Element (Boyer–Moore cũng là "giữ ứng viên + bộ đếm").

## 14. Biến thể

### Biến thể 1 — Trả về **chỉ số** các hình đạt max

In [21]:
def good_indices(rects):
    best, idx = 0, []
    for i, (l, w) in enumerate(rects):
        s = min(l, w)
        if s > best:
            best, idx = s, [i]
        elif s == best:
            idx.append(i)
    return idx

print(good_indices([[5, 8], [3, 9], [5, 12], [16, 5]]))

[0, 2, 3]


### Biến thể 2 — Đếm hình đạt cạnh vuông **nhỏ nhất**

Đối xứng: đổi `>` thành `<`, khởi tạo bằng `float('inf')` (vô cực dương — lớn hơn mọi số).

In [22]:
def count_smallest_square(rects):
    best, count = float("inf"), 0
    for l, w in rects:
        s = min(l, w)
        if s < best:
            best, count = s, 1
        elif s == best:
            count += 1
    return count

print(float("inf") > 10**18, count_smallest_square([[5, 8], [3, 9], [5, 12], [16, 5]]))

True 1


### Biến thể 3 — Với mỗi truy vấn `k`, bao nhiêu hình cắt được vuông cạnh `k`?

Sắp `sides` tăng dần; số hình có `s >= k` = `n - bisect_left(sides, k)`. `bisect_left` (module `bisect`) tìm vị trí chèn đầu tiên trong list đã sắp — `O(log n)`.

In [23]:
from bisect import bisect_left

def count_can_cut(rects, queries):
    sides = sorted(min(r) for r in rects)
    n = len(sides)
    return [n - bisect_left(sides, k) for k in queries]

print(count_can_cut([[5, 8], [3, 9], [5, 12], [16, 5]], [1, 3, 4, 5, 6]))   # [4, 4, 3, 3, 0]

[4, 4, 3, 3, 0]


## 15. Bài tập (tăng dần)

> Hàm bài tập chỉ có `pass`, **không được gọi** khi chạy notebook.

### Bài 1 — Tính tay
`[[7, 3], [4, 9], [8, 4], [2, 10], [4, 5]]` → `sides`, `maxLen`, đáp án?

In [24]:
def exercise_1():
    # TODO: trả về (sides, maxLen, đáp án)
    pass

# assert exercise_1() == ([3, 4, 4, 2, 4], 4, 3)

### Bài 2 — Viết `square_sides(rects)` bằng vòng `for` (không list comprehension)

In [25]:
def square_sides(rects: List[List[int]]) -> List[int]:
    # TODO
    pass

# assert square_sides([[5, 8], [16, 5]]) == [5, 5]

### Bài 3 — Điền chỗ trống lời giải một lượt

In [26]:
def one_pass_fill(rects: List[List[int]]) -> int:
    max_len, count = 0, 0
    for l, w in rects:
        s = min(l, w)
        # TODO 1: nếu s > max_len -> ?
        # TODO 2: elif s == max_len -> ?
        pass
    return count

# assert one_pass_fill([[2, 3], [3, 7], [4, 3], [3, 7]]) == 3

### Bài 4 — Đếm số phần tử bằng max trong một list bất kỳ (một lượt)

In [27]:
def count_max(nums: List[int]) -> int:
    # TODO: nums có thể chứa số âm -> khởi tạo thế nào?
    pass

# assert count_max([3, 1, 3, -2, 3]) == 3 and count_max([-5, -1, -1]) == 2

### Bài 5 — Biến thể: cạnh vuông lớn thứ hai
Trả về số hình đạt **giá trị cạnh lớn thứ hai khác nhau** (trả 0 nếu chỉ có một giá trị).

In [28]:
def count_second_largest(rects: List[List[int]]) -> int:
    # TODO
    pass

# assert count_second_largest([[5, 8], [3, 9], [5, 12], [16, 5]]) == 1
# assert count_second_largest([[4, 7], [9, 4]]) == 0

### Bài 6 — Mini challenge: ghép hai hình
Được chọn **hai hình khác nhau** đặt cạnh nhau theo chiều dài (ghép `[l1, w1]` và `[l2, w2]` thành `[l1 + l2, min(w1, w2)]` sau khi **được xoay** mỗi hình tuỳ ý). Cạnh hình vuông lớn nhất cắt được từ một cách ghép tốt nhất là bao nhiêu? Viết brute force `O(n²)`.

In [29]:
def best_square_after_join(rects: List[List[int]]) -> int:
    # TODO: thử mọi cặp i < j và mọi cách xoay (2 x 2)
    pass

# assert best_square_after_join([[1, 5], [1, 5]]) == 2
# assert best_square_after_join([[3, 9], [5, 6]]) == 6

## 16. Gợi ý 3 mức

**Bài 1:** (1) Lấy `min` từng cặp. (2) `sides = [3, 4, 4, 2, 4]`. (3) `maxLen = 4`, xuất hiện 3 lần.

**Bài 2:** (1) List rỗng `res`. (2) `res.append(min(l, w))`. (3) `return res`.

**Bài 3:** (1) Max mới ⇒ reset. (2) `max_len, count = s, 1`. (3) `count += 1`.

**Bài 4:** (1) Không thể khởi tạo `0` vì có số âm. (2) Dùng `float('-inf')` hoặc `nums[0]`. (3) Cùng khung `(best, count)`.

**Bài 5:** (1) Tìm các giá trị khác nhau. (2) `vals = sorted(set(sides), reverse=True)`. (3) Nếu `len(vals) < 2` trả 0, ngược lại `sides.count(vals[1])`.

**Bài 6:** (1) Mỗi hình có 2 hướng: `(l, w)` hoặc `(w, l)`. (2) Ghép theo chiều thứ nhất: `[a1 + a2, min(b1, b2)]`. (3) Cạnh vuông = `min(a1 + a2, min(b1, b2))`, lấy max trên mọi cặp và mọi hướng.

## 17. Đáp án tham khảo

In [30]:
def ans_exercise_1():
    rects = [[7, 3], [4, 9], [8, 4], [2, 10], [4, 5]]
    sides = [min(l, w) for l, w in rects]
    m = max(sides)
    return (sides, m, sides.count(m))

def ans_square_sides(rects):
    res = []
    for l, w in rects:
        res.append(min(l, w))
    return res

def ans_one_pass_fill(rects):
    max_len, count = 0, 0
    for l, w in rects:
        s = min(l, w)
        if s > max_len:
            max_len, count = s, 1
        elif s == max_len:
            count += 1
    return count

def ans_count_max(nums):
    best, count = float("-inf"), 0
    for x in nums:
        if x > best:
            best, count = x, 1
        elif x == best:
            count += 1
    return count

def ans_count_second_largest(rects):
    sides = [min(r) for r in rects]
    vals = sorted(set(sides), reverse=True)
    return sides.count(vals[1]) if len(vals) >= 2 else 0

def ans_best_square_after_join(rects):
    best = 0
    n = len(rects)
    for i in range(n):
        for j in range(i + 1, n):
            for a1, b1 in (rects[i], rects[i][::-1]):
                for a2, b2 in (rects[j], rects[j][::-1]):
                    best = max(best, min(a1 + a2, min(b1, b2)))
    return best

assert ans_exercise_1() == ([3, 4, 4, 2, 4], 4, 3)
assert ans_square_sides([[5, 8], [16, 5]]) == [5, 5]
assert ans_one_pass_fill([[2, 3], [3, 7], [4, 3], [3, 7]]) == 3
assert ans_count_max([3, 1, 3, -2, 3]) == 3 and ans_count_max([-5, -1, -1]) == 2
assert ans_count_second_largest([[5, 8], [3, 9], [5, 12], [16, 5]]) == 1
assert ans_count_second_largest([[4, 7], [9, 4]]) == 0
assert ans_best_square_after_join([[1, 5], [1, 5]]) == 2
assert ans_best_square_after_join([[3, 9], [5, 6]]) == 6
for _ in range(500):
    nums = [random.randint(-5, 5) for _ in range(random.randint(1, 10))]
    assert ans_count_max(nums) == nums.count(max(nums))
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def countGoodRectangles(self, rectangles):
        max_len, count = 0, 0
        for l, w in rectangles:
            s = min(l, w)                  # cạnh vuông lớn nhất của hình này
            if s > max_len:                # max mới -> reset
                max_len, count = s, 1
            elif s == max_len:             # thêm một hình đạt max
                count += 1
        return count
```

**Khung tư duy:** rút mỗi đối tượng về một số → "đếm số phần tử bằng max" → một lượt `(best, count)`.

**Checklist trước khi submit**
- [ ] Dùng `min(l, w)` (không phải `max`, không phải diện tích)?
- [ ] Reset `count = 1` khi có max mới?
- [ ] Dùng `elif` cho nhánh bằng nhau?
- [ ] Giá trị khởi tạo `max_len` nhỏ hơn mọi cạnh có thể (ở đây cạnh ≥ 1 nên 0 an toàn)?
- [ ] Tên method `countGoodRectangles`, trả về `int`?